In [3]:
# ============================================================
# TRI^p (context-aware) for 8 datasets
# - Flow/IoT datasets scored vs provided baselines (enterprise / iot / encrypted)
# - UNR-IDD scored via LOSO (builds fold baselines internally; no leakage)
# ============================================================

import os, glob, math, yaml
from datetime import datetime
from typing import Dict, List, Tuple, Optional
import numpy as np
import pandas as pd

# ---------- collectors for service-aware diagnostics ----------
OVERLAP_ROWS: list = []   # rows: dataset, service, role('test'|'baseline'|'common'), share
CONTRIB_ROWS: list = []   # rows: dataset, service, weight, stat_used, sim, n

# Optional: print per-service summaries for these datasets
DEBUG_SERVICES = set()  # e.g., {"ToN-IoT", "HIKARI-2021"}

# ------------------ CONFIG (EDIT) ------------------
BASELINES = {
    "enterprise": r"baselines/general_enterprise_nids_baseline_20251016.yaml",
    "iot":        r"baselines/iot_gateway_nids_baseline_20251017.yaml",
    "encrypted_flow_only_ids": r"baselines/encrypted_flow_only_ids_baseline_20251129.yaml",
    # NOTE: no counters baseline for UNR-IDD; LOSO builds fold baselines on the fly
}

DATASETS = {
    "NSL-KDD":           r"E:/Datasets/NSL-KDD/NSL-KDD.csv",
    "UNSW-NB15":         r"E:/Datasets/UNSWNB15/unsw.csv",
    "CIC-IDS2017":       r"E:/Datasets/CICIDS2017/combined_file.csv",
    "CSE-CIC-IDS2018":   r"F:/Datasets/CSE-CIC-IDS2018/CSE-CIC-IDS2018.csv",
    "ToN-IoT":           r"F:/Datasets/ToN_IoT/train_test_network.csv",
    "BoT-IoT":           r"F:/Datasets/Bot_IoT/Bot_IoT.csv",
    "HIKARI-2021":       r"E:/Datasets/HIKARI2021/ALLFLOWMETER_HIKARI2021.csv",
    "UNR-IDD":           r"F:/Datasets/UNR-IDD/UNR-IDD.csv",
}

PROFILE_MAP = {
    "NSL-KDD":         "enterprise",
    "UNSW-NB15":       "enterprise",
    "CIC-IDS2017":     "enterprise",
    "CSE-CIC-IDS2018": "enterprise",
    "ToN-IoT":         "iot",
    "BoT-IoT":         "iot",    
    "HIKARI-2021":     "encrypted_flow_only_ids",
    "UNR-IDD":         "counters",  # handled specially via LOSO below
}

WEIGHTS = {
    "enterprise": dict(alpha=0.4, beta=0.4, gamma=0.2),
    "iot":        dict(alpha=0.4, beta=0.4, gamma=0.2),
    "encrypted_flow_only_ids": dict(alpha=0.4, beta=0.4, gamma=0.2),
    "counters":   dict(alpha=0.7, beta=0.3, gamma=0.0),  # used inside LOSO
}

# UNR-IDD LOSO settings
COUNTERS_TARGET_WINDOW_S = 60
COUNTERS_TOLERANCE_S     = 10
N_BINS = 50
WINSOR = (0.005, 0.995)
# ---------------------------------------------------

# ----------------- utilities -----------------
def _match_paths(path_or_glob: str) -> List[str]:
    if os.path.isfile(path_or_glob):
        return [path_or_glob]
    return sorted(glob.glob(path_or_glob, recursive=True))

def _ci(df) -> Dict[str,str]:
    return {str(c).lower().strip(): c for c in df.columns}

def _col(nm: Dict[str,str], *names) -> Optional[str]:
    for name in names:
        got = nm.get(name.lower())
        if got: return got
    return None

def _to_num(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series.astype(str).str.replace(",","").str.strip(), errors="coerce")

def _winsor(series: pd.Series, q_low=WINSOR[0], q_high=WINSOR[1]) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce").dropna().astype(float)
    if s.empty: return s
    lo, hi = s.quantile([q_low, q_high])
    return s.clip(lower=lo, upper=hi)

def _log10_cdf_on_bins(series: pd.Series, bins_log10: List[float]) -> List[float]:
    s = _winsor(series)
    s = s[s > 0]
    if s.empty: return [0.0]*(len(bins_log10)-1)
    logv = np.log10(s.values)
    hist, _ = np.histogram(logv, bins=np.array(bins_log10), density=True)
    if np.sum(hist) == 0: return [0.0]*(len(bins_log10)-1)
    return (np.cumsum(hist)/np.sum(hist)).tolist()

def _lin_cdf_on_bins(series: pd.Series, bins: List[float]) -> List[float]:
    s = _winsor(series)
    if s.empty: return [0.0]*(len(bins)-1)
    hist, _ = np.histogram(s.values, bins=np.array(bins), density=True)
    if np.sum(hist) == 0: return [0.0]*(len(bins)-1)
    return (np.cumsum(hist)/np.sum(hist)).tolist()

def ks_distance(cdf_ref: List[float], cdf_test: List[float]) -> float:
    if not cdf_ref or len(cdf_ref) != len(cdf_test): return 1.0
    return float(np.max(np.abs(np.array(cdf_ref) - np.array(cdf_test))))

def tv_distance(p_ref: Dict[str,float], p_test: Dict[str,float]) -> float:
    keys = set(p_ref.keys()) | set(p_test.keys())
    return 0.5 * sum(abs(p_ref.get(k,0.0) - p_test.get(k,0.0)) for k in keys)

def load_csv(path_or_glob: str) -> pd.DataFrame:
    frames = []
    for p in _match_paths(path_or_glob):
        try:
            frames.append(pd.read_csv(p, low_memory=False))
        except Exception as e:
            print(f"[WARN] Could not read {p}: {e}")
    if not frames:
        raise FileNotFoundError(f"No files matched for {path_or_glob}")
    return pd.concat(frames, ignore_index=True)

def load_baseline(path: str) -> Dict:
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)

def norm_proto(v):
    v = str(v).strip().lower()
    if v in ("6","tcp"): return "tcp"
    if v in ("17","udp"): return "udp"
    if v in ("1","icmp"): return "icmp"
    return v or "unknown"

_PORT2SVC = {
    # Core
    20:"ftp-data", 21:"ftp", 22:"ssh", 23:"telnet", 25:"smtp",
    53:"dns", 67:"dhcp", 68:"dhcp", 69:"tftp",
    80:"http", 110:"pop3", 123:"ntp", 135:"dce_rpc", 137:"netbios_ns", 138:"netbios_dgm",
    139:"smb", 143:"imap", 161:"snmp", 389:"ldap", 443:"https",
    445:"smb", 465:"smtp", 500:"isakmp", 520:"rip",
    587:"submission", 993:"imaps", 995:"pop3s",

    # Web variants
    8000:"http", 8001:"http", 8080:"http-alt", 8443:"https", 8888:"http",

    # DB / remote desktop
    1433:"mssql", 1521:"oracle", 27017:"mongodb", 3306:"mysql", 3389:"rdp",

    # VoIP / streaming
    5060:"sip", 554:"rtsp", 8554:"rtsp",

    # IoT / discovery
    5353:"mdns", 5355:"llmnr", 1900:"ssdp", 3702:"ws-discovery",
    1883:"mqtt", 8883:"mqtts",

    # Security/tunneling
    853:"dot", 4500:"ipsec-natt",
}
def infer_service_from_port(series: pd.Series):
    s = pd.to_numeric(series, errors="coerce")
    return s.map(lambda x: _PORT2SVC.get(int(x), None) if pd.notna(x) else None)

# --- service canonicalization helpers ---
_SERVICE_ALIAS = {
    "http-alt":"http",
    "www":"http",
    "mdns":"dns",
    "llmnr":"dns",     # treat as DNS-family for timing
    "submission":"smtp",
    "pop3s":"pop3",
    "imaps":"imap",
    "ssl":"https",     # some datasets label TLS like this
    "https-alt":"https",
}

def canonicalize_service_name(s: str) -> str:
    if not isinstance(s, str): return s
    x = s.strip().lower()
    if x in ("", "-", "none", "nan"): return None
    return _SERVICE_ALIAS.get(x, x)

def canonicalize_service_series(series: pd.Series) -> pd.Series:
    if series is None: 
        return series
    out = series.astype(str).map(canonicalize_service_name)
    return out.where(out.notna(), other=None)

# --- helpers used by service-aware TR_tmp ---

def _service_hist(series: Optional[pd.Series]) -> dict:
    """Return normalized histogram {svc: share} (expects canonicalized service labels)."""
    if series is None:
        return {}
    s = series.dropna().astype(str)
    s = s[(s != "") & (s != "-") & (s != "none") & (s != "nan")]
    if s.empty:
        return {}
    return s.value_counts(normalize=True).to_dict()

def _ks_similarity(sample_vals: pd.Series, ref_bins: list, ref_cdf: list) -> float:
    """1 - KS distance between sample empirical CDF (on ref_bins) and ref CDF."""
    s = pd.to_numeric(sample_vals, errors="coerce").dropna()
    s = s[s > 0]
    if s.empty or not ref_bins or not ref_cdf:
        return np.nan
    logv = np.log10(s.values)
    hist, _ = np.histogram(logv, bins=np.array(ref_bins), density=True)
    if np.sum(hist) == 0:
        return np.nan
    cdf_s = np.cumsum(hist) / np.sum(hist)
    ks = np.max(np.abs(cdf_s - np.array(ref_cdf)))
    return float(1.0 - ks)


def mix(series: Optional[pd.Series]) -> Optional[Dict[str,float]]:
    if series is None: return None
    s = series.dropna().astype(str).str.lower()
    s = s[(s!='-') & (s!='none') & (s!='')]
    vc = s.value_counts()
    if vc.sum() == 0: return None
    return (vc / vc.sum()).to_dict()

# -------------- normalizers (flow/IoT) --------------
def _dur_to_seconds(series: pd.Series) -> pd.Series:
    s = _to_num(series)
    if s.dropna().median() and s.dropna().median() > 1e5:
        s = s/1e6  # micros → seconds
    return s

def norm_nsl_kdd(df: pd.DataFrame) -> pd.DataFrame:
    nm = _ci(df)
    dur = _to_num(df[nm["duration"]]) if "duration" in nm else np.nan
    b  = _to_num(df[nm["src_bytes"]]) if "src_bytes" in nm else 0
    b += _to_num(df[nm["dst_bytes"]]) if "dst_bytes" in nm else 0
    proto = df[nm["protocol_type"]].map(norm_proto) if "protocol_type" in nm else "unknown"
    service = df[nm["service"]].astype(str).str.lower() if "service" in nm else None
    return pd.DataFrame({"bytes":b, "pkts":np.nan, "duration":dur, "start_ts":np.nan,
                         "proto":proto, "service":service})

def norm_unsw(df: pd.DataFrame) -> pd.DataFrame:
    nm = _ci(df)
    sbytes = _to_num(df[nm["sbytes"]]); dbytes = _to_num(df[nm["dbytes"]])
    spkts  = _to_num(df[nm["spkts"]]);  dpkts  = _to_num(df[nm["dpkts"]])
    dur    = _to_num(df[nm["dur"]])     # seconds

    bytes_ = sbytes.fillna(0) + dbytes.fillna(0)
    pkts   = spkts.fillna(0) + dpkts.fillna(0)

    denom = (pkts.where(pkts >= 2, 2) - 1)  # IAT proxy
    flow_iat_mean_vals = (dur / denom).replace([np.inf, -np.inf], np.nan)

    proto = df[nm["proto"]].map(norm_proto) if "proto" in nm else "unknown"
    service = df[nm["service"]].astype(str).str.lower() if "service" in nm else None

    return pd.DataFrame({
        "bytes": bytes_,
        "pkts": pkts,
        "duration": dur,
        "start_ts": np.nan,
        "proto": proto,
        "service": service,
        "flow_iat_mean_vals": flow_iat_mean_vals,
    })

def _infer_proto_cic2017(df: pd.DataFrame, nm: dict) -> pd.Series:
    flag_cols = []
    for k in ["syn flag count","ack flag count","rst flag count","fin flag count",
              "urg flag count","ece flag count","psh flag count"]:
        if k in nm: flag_cols.append(nm[k])
    proto = pd.Series("unknown", index=df.index, dtype="object")
    if flag_cols:
        flags_sum = df[flag_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
        proto = np.where(flags_sum > 0, "tcp", proto)
    udp_ports = {53,67,68,69,123,137,138,161,500,520,1900,4500,5353,5683,27015}
    tcp_ports = {20,21,22,23,25,80,110,143,443,465,587,993,995,3306,3389,8080,8443,27017}
    if "destination port" in nm:
        dport = pd.to_numeric(df[nm["destination port"]], errors="coerce")
        unk_mask = (proto == "unknown")
        udp_mask = dport.isin(list(udp_ports)) & unk_mask
        tcp_mask = dport.isin(list(tcp_ports)) & unk_mask
        proto = pd.Series(proto).mask(udp_mask, "udp").mask(tcp_mask, "tcp")
    return pd.Series(proto, index=df.index, dtype="object")

def _normalize_iat_units_to_seconds(series: pd.Series) -> pd.Series:
    x = pd.to_numeric(series, errors="coerce")
    x_pos = x[x > 0]
    if x_pos.empty:
        return x
    q90 = x_pos.quantile(0.90)
    if q90 > 1e5:       # microseconds
        return x / 1e6
    elif q90 > 1e3:     # milliseconds
        return x / 1e3
    else:
        return x        # seconds

def norm_cic2017(df: pd.DataFrame) -> pd.DataFrame:
    nm = _ci(df)
    b = _to_num(df[nm["total length of fwd packets"]]) + _to_num(df[nm["total length of bwd packets"]])
    p = _to_num(df[nm["total fwd packets"]]) + _to_num(df[nm["total backward packets"]])
    dur = _dur_to_seconds(df[nm["flow duration"]])
    iat_mean = _normalize_iat_units_to_seconds(df[nm["flow iat mean"]]) if "flow iat mean" in nm else pd.Series(np.nan, index=df.index)
    proto = _infer_proto_cic2017(df, nm)
    service = infer_service_from_port(df[nm["destination port"]]) if "destination port" in nm else np.nan
    return pd.DataFrame({
        "bytes": b, "pkts": p, "duration": dur, "start_ts": np.nan,
        "proto": proto, "service": service, "flow_iat_mean_vals": iat_mean
    })

def norm_cse_cic_2018(df: pd.DataFrame) -> pd.DataFrame:
    nm = _ci(df)
    if "totlen fwd pkts" in nm and "totlen bwd pkts" in nm:
        b = _to_num(df[nm["totlen fwd pkts"]]) + _to_num(df[nm["totlen bwd pkts"]])
    else:
        pkts_total = _to_num(df[nm["tot fwd pkts"]]) + _to_num(df[nm["tot bwd pkts"]])
        pkt_size_col = _ci(df).get("pkt len mean") or _ci(df).get("pkt size avg")
        pkt_size_avg = _to_num(df.get(pkt_size_col, pd.Series(np.nan, index=df.index)))
        b = pkts_total * pkt_size_avg
    p   = _to_num(df[nm["tot fwd pkts"]]) + _to_num(df[nm["tot bwd pkts"]])
    dur = _dur_to_seconds(df[nm["flow duration"]])
    iat_mean = _normalize_iat_units_to_seconds(df[nm["flow iat mean"]]) if "flow iat mean" in nm else pd.Series(np.nan, index=df.index)
    proto = df[nm["protocol"]].map(norm_proto) if "protocol" in nm else np.nan
    service = infer_service_from_port(df[nm["dst port"]]) if "dst port" in nm else np.nan
    return pd.DataFrame({
        "bytes": b, "pkts": p, "duration": dur, "start_ts": np.nan,
        "proto": proto, "service": service, "flow_iat_mean_vals": iat_mean
    })

def norm_ton_iot(df: pd.DataFrame) -> pd.DataFrame:
    nm = _ci(df)
    s_pk  = _to_num(df[nm["src_pkts"]]) if "src_pkts" in nm else 0
    d_pk  = _to_num(df[nm["dst_pkts"]]) if "dst_pkts" in nm else 0
    pkts  = (s_pk.fillna(0) + d_pk.fillna(0))
    bytes_= _to_num(df[nm["src_bytes"]]).fillna(0) + _to_num(df[nm["dst_bytes"]]).fillna(0)
    dur   = _to_num(df[nm["duration"]])
    denom = (pkts.where(pkts >= 2, 2) - 1)
    flow_iat_mean_vals = (dur / denom).replace([np.inf, -np.inf], np.nan)
    proto = df[nm["proto"]].map(norm_proto) if "proto" in nm else "unknown"
    svc   = df[nm["service"]].astype(str).str.lower() if "service" in nm else infer_service_from_port(df.get(nm.get("dst_port",""), np.nan))
    return pd.DataFrame({
        "bytes": bytes_, "pkts": pkts, "duration": dur, "start_ts": np.nan,
        "proto": proto, "service": svc, "flow_iat_mean_vals": flow_iat_mean_vals
    }).replace([np.inf,-np.inf], np.nan)

def norm_bot_iot(df: pd.DataFrame) -> pd.DataFrame:
    nm = _ci(df)

    # bytes
    b = _to_num(df[nm.get("sbytes","bytes")]).fillna(0) + _to_num(df[nm.get("dbytes","bytes")]).fillna(0)

    # pkts
    if "spkts" in nm and "dpkts" in nm:
        p = _to_num(df[nm["spkts"]]).fillna(0) + _to_num(df[nm["dpkts"]]).fillna(0)
    else:
        p = _to_num(df.get(nm.get("pkts","pkts"), pd.Series(np.nan, index=df.index))).fillna(0)

    # duration
    dur = _to_num(df.get(nm.get("dur","dur"), pd.Series(np.nan, index=df.index)))

    # optional start timestamp (some Bot-IoT dumps have stime)
    ts = _to_num(df.get(nm.get("stime","stime"), pd.Series(np.nan, index=df.index)))
    if ts.dropna().median() and ts.dropna().median() > 1e11:
        ts = ts / 1000.0  # ms → s

    # protocol
    proto = df.get(nm.get("proto","proto"), pd.Series("unknown", index=df.index))
    proto = proto.map(norm_proto)

    # --- NEW: infer service from destination port ---
    dport_col = nm.get("dport") or nm.get("dst_port") or nm.get("destination port") or nm.get("responp") or nm.get("dstp")
    if dport_col:
        service = infer_service_from_port(df[dport_col])
    else:
        service = pd.Series([None]*len(df), index=df.index)

    out = pd.DataFrame({
        "bytes": b, "pkts": p, "duration": dur, "start_ts": ts,
        "proto": proto, "service": service
    }).replace([np.inf,-np.inf], np.nan)

    # canonicalize service labels now
    out["service"] = canonicalize_service_series(out["service"])
    return out


def norm_hikari(df: pd.DataFrame) -> pd.DataFrame:
    nm = _ci(df)
    p = _to_num(df[nm.get("fwd_pkts_tot","fwd_pkts_tot")]).fillna(0) + \
        _to_num(df[nm.get("bwd_pkts_tot","bwd_pkts_tot")]).fillna(0)
    if "fwd_subflow_bytes" in nm and "bwd_subflow_bytes" in nm:
        b = _to_num(df[nm["fwd_subflow_bytes"]]).fillna(0) + _to_num(df[nm["bwd_subflow_bytes"]]).fillna(0)
    else:
        f_pay = _to_num(df[nm.get("fwd_pkts_payload.tot","fwd_pkts_payload.tot")]).fillna(0)
        b_pay = _to_num(df[nm.get("bwd_pkts_payload.tot","bwd_pkts_payload.tot")]).fillna(0)
        b = f_pay + b_pay
    dur = _to_num(df[nm.get("flow_duration","flow_duration")])
    service = infer_service_from_port(df[nm.get("responp","responp")]) if "responp" in nm else np.nan
    flag_cols = [c for k,c in nm.items() if k in {
        "flow_syn_flag_count","flow_ack_flag_count","flow_fin_flag_count",
        "flow_rst_flag_count","flow_urg_flag_count","flow_ece_flag_count","flow_psh_flag_count"}]
    if flag_cols:
        flags_sum = df[flag_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
        proto = np.where(flags_sum > 0, "tcp", "unknown")
    else:
        proto = "unknown"
    if "flow_iat.avg" in nm:
        iat_mean_vals = _normalize_iat_units_to_seconds(df[nm["flow_iat.avg"]])
    elif "flow_iat_mean" in nm:
        iat_mean_vals = _normalize_iat_units_to_seconds(df[nm["flow_iat_mean"]])
    else:
        denom = (p.where(p >= 2, 2) - 1)
        iat_mean_vals = (dur / denom).replace([np.inf, -np.inf], np.nan)
    return pd.DataFrame({
        "bytes": b, "pkts": p, "duration": dur, "start_ts": np.nan,
        "proto": pd.Series(proto, index=df.index),
        "service": service,
        "flow_iat_mean_vals": iat_mean_vals
    }).replace([np.inf, -np.inf], np.nan)

# ---------- Service-aware temporal ----------
def score_temporal_service_aware(
    test_df: pd.DataFrame,
    metrics: dict,
    debug: bool = False,
    dataset_name: Optional[str] = None
) -> float:
    """
    Service-aware TR_tmp with:
      - canonicalized service labels
      - min-N guard per service
      - test-only weights over common services
      - adaptive blend: mass_common_test * svc_sim + (1 - mass_common_test) * global_sim
      - ALWAYS returns a float (np.nan when unavailable)
    """
    name = dataset_name or ""

    # what temporal stats can we use?
    has_start_ts = ("start_ts" in test_df.columns) and pd.notna(test_df["start_ts"]).any()
    has_mean_iat = ("flow_iat_mean_vals" in test_df.columns) and pd.notna(test_df["flow_iat_mean_vals"]).any()
    use_iat  = has_start_ts and ("iat_flows" in metrics)
    use_mean = (not use_iat) and has_mean_iat and ("flow_iat_mean" in metrics)

    # ---------- GLOBAL SIM (computed once) ----------
    def _ks_similarity(sample_vals: pd.Series, ref_bins: list, ref_cdf: list) -> float:
        s = pd.to_numeric(sample_vals, errors="coerce").dropna()
        s = s[s > 0]
        if s.empty or not ref_bins or not ref_cdf:
            return np.nan
        logv = np.log10(s.values)
        hist, _ = np.histogram(logv, bins=np.array(ref_bins), density=True)
        if np.sum(hist) == 0:
            return np.nan
        cdf_s = np.cumsum(hist) / np.sum(hist)
        ks = np.max(np.abs(cdf_s - np.array(ref_cdf)))
        return float(1.0 - ks)
    
    global_sim = np.nan
    if use_iat:
        ts_all = pd.to_numeric(test_df["start_ts"], errors="coerce").dropna().sort_values()
        iat_all = ts_all.diff().dropna(); iat_all = iat_all[iat_all > 0]
        if not iat_all.empty:
            global_sim = _ks_similarity(iat_all, metrics["iat_flows"]["bins_log10"], metrics["iat_flows"]["cdf"])
    elif use_mean:
        vals_all = pd.to_numeric(test_df["flow_iat_mean_vals"], errors="coerce").dropna()
        if not vals_all.empty:
            global_sim = _ks_similarity(vals_all, metrics["flow_iat_mean"]["bins_log10"], metrics["flow_iat_mean"]["cdf"])
    
    # ---------- baseline per-service temporal (canonicalize + merge) ----------
    def canonicalize_service_name(s: str) -> Optional[str]:
        if not isinstance(s, str): return None
        x = s.strip().lower()
        if x in ("", "-", "none", "nan"): return None
        return _SERVICE_ALIAS.get(x, x)
    
    def canonicalize_service_series(series: pd.Series) -> pd.Series:
        if series is None:
            return series
        out = series.astype(str).map(canonicalize_service_name)
        return out.where(out.notna(), other=None)
    
    test_services = canonicalize_service_series(test_df.get("service"))
    svc_hist_test = {}
    if test_services is not None:
        s = test_services.dropna().astype(str)
        s = s[(s != "-") & (s != "none") & (s != "")]
        if not s.empty:
            svc_hist_test = s.value_counts(normalize=True).to_dict()
    
    raw_ref = metrics.get("temporal_by_service", {}) or {}
    svc_ref = {}
    for s_raw, node in raw_ref.items():
        s = canonicalize_service_name(s_raw)
        if s is None:
            continue
        if s not in svc_ref:
            svc_ref[s] = dict(node)
        else:
            # merge shares and prefer the node with more 'count' for hist fields
            svc_ref[s]["share"] = float(svc_ref[s].get("share", 0.0)) + float(node.get("share", 0.0))
            c_old = int(svc_ref[s].get("count", 0))
            c_new = int(node.get("count", 0))
            if c_new > c_old:
                for k in ("iat_flows", "flow_iat_mean"):
                    if k in node:
                        svc_ref[s][k] = node[k]
                svc_ref[s]["count"] = c_new
    
    svc_hist_ref = {s: float(v.get("share", 0.0)) for s, v in svc_ref.items()} if svc_ref else {}
    common = sorted(set(svc_hist_test.keys()) & set(svc_hist_ref.keys()))
    
    # collect overlap rows (for CSV)
    if svc_hist_test:
        for s, v in svc_hist_test.items():
            OVERLAP_ROWS.append({"dataset": name, "service": s, "role": "test", "share": float(v)})
    if svc_hist_ref:
        for s, v in svc_hist_ref.items():
            OVERLAP_ROWS.append({"dataset": name, "service": s, "role": "baseline", "share": float(v)})
    for s in common:
        OVERLAP_ROWS.append({"dataset": name, "service": s, "role": "common",
                             "share": float(min(svc_hist_test.get(s, 0.0), svc_hist_ref.get(s, 0.0)))})
    
    # debug print
    if debug:
        print(f"\n[TR_tmp:Service overlap summary] {name}")
        if not svc_hist_test:
            print("  Test dataset: no valid service labels found.")
        else:
            top_test = sorted(svc_hist_test.items(), key=lambda x: x[1], reverse=True)[:10]
            print("  Top test services:")
            for s, v in top_test:
                print(f"    {s:15s} {v:6.3f}  {'in baseline' if s in svc_hist_ref else 'not in baseline'}")
        if svc_hist_ref:
            top_ref = sorted(svc_hist_ref.items(), key=lambda x: x[1], reverse=True)[:10]
            print("  Top baseline services:")
            for s, v in top_ref:
                print(f"    {s:15s} {v:6.3f}  {'✓' if s in common else ' '}")
        else:
            print("  Baseline: no service temporal reference.")
        if not common:
            print("  ⚠ No common services between dataset and baseline.")
        else:
            print(f"  Common services ({len(common)}): {', '.join(common[:8])}{'…' if len(common) > 8 else ''}")
        print()
    
    # if no per-service reference or no overlap, return global_sim (could be NaN)
    if not svc_ref or not common:
        try:
            return float(global_sim)
        except Exception:
            return np.nan
    
    # ---------- Min-N guard + test-only weights over common ----------
    MIN_N = max(200, int(0.00005 * len(test_df)))  # 0.005% of rows, floor 200
    svc_series = test_services.astype(str)
    svc_counts = {s: int((svc_series == s).sum()) for s in common}
    kept = [s for s in common if svc_counts.get(s, 0) >= MIN_N]
    if not kept:
        try:
            return float(global_sim)
        except Exception:
            return np.nan
    
    mass_common_test = sum(svc_hist_test.get(s, 0.0) for s in kept)
    if mass_common_test <= 0:
        try:
            return float(global_sim)
        except Exception:
            return np.nan
    
    # ---------- Per-service sims on kept services ----------
    sims = []
    details = []
    for s in kept:
        node = svc_ref[s]
        w_s = float(svc_hist_test.get(s, 0.0)) / mass_common_test
        sim = np.nan
        stat_used = None
        mask = (svc_series == s)
    
        if use_iat and ("iat_flows" in node):
            ts = pd.to_numeric(test_df.loc[mask, "start_ts"], errors="coerce").dropna().sort_values()
            iat = ts.diff().dropna(); iat = iat[iat > 0]
            sim = _ks_similarity(iat, node["iat_flows"]["bins_log10"], node["iat_flows"]["cdf"])
            stat_used = "iat_flows"
        elif use_mean and ("flow_iat_mean" in node):
            vals = pd.to_numeric(test_df.loc[mask, "flow_iat_mean_vals"], errors="coerce").dropna()
            sim = _ks_similarity(vals, node["flow_iat_mean"]["bins_log10"], node["flow_iat_mean"]["cdf"])
            stat_used = "flow_iat_mean"
    
        if not np.isnan(sim):
            sims.append(w_s * sim)
            CONTRIB_ROWS.append({
                "dataset": name, "service": "__blend__", "weight": float(mass_common_test),
                "stat_used": "blend_mass", "sim": float(global_sim) if not np.isnan(global_sim) else np.nan,
                "n": int(len(test_df))
            })
    
            details.append((s, w_s, stat_used or "", float(sim), int(mask.sum())))
    
    svc_sim = float(np.sum(sims)) if sims else np.nan
    
    if debug and details:
        print(f"[TR_tmp:svc-aware] {name} — contributing services (w, stat, sim, n):")
        for s, w, stat_used, sim, n in sorted(details, key=lambda z: z[1], reverse=True):
            print(f"  {s:15s} w={w:6.3f}  stat={stat_used:12s}  sim={sim:6.3f}  n={n}")
    
    # ---------- Adaptive blend ----------
    if np.isnan(svc_sim):
        tr_tmp = global_sim
    elif np.isnan(global_sim):
        tr_tmp = svc_sim
    else:
        k = 1.5  # >1 favors per-service more
        w = mass_common_test ** k
        tr_tmp = w * svc_sim + (1.0 - w) * global_sim
    
    
    # ALWAYS return a float (or NaN)
    try:
        return float(tr_tmp)
    except Exception:
        return np.nan


# -------------- flow/IoT scoring --------------
def score_flow_like(df_norm: pd.DataFrame, baseline: Dict, weights: Dict[str,float], dataset_name: str) -> Dict[str,float]:
    m = baseline["metrics"]

    # TR_vol: bytes, pkts, duration
    vol_parts = []; vol_wts=[]
    if "bytes_per_flow" in m and "bins_log10" in m["bytes_per_flow"]:
        tb = _log10_cdf_on_bins(df_norm["bytes"], m["bytes_per_flow"]["bins_log10"])
        db = ks_distance(m["bytes_per_flow"]["cdf"], tb); vol_parts.append(1.0 - db); vol_wts.append(1.0)
    if "pkts_per_flow" in m and "bins_log10" in m["pkts_per_flow"] and df_norm["pkts"].notna().any():
        tp = _log10_cdf_on_bins(df_norm["pkts"], m["pkts_per_flow"]["bins_log10"])
        dp = ks_distance(m["pkts_per_flow"]["cdf"], tp); vol_parts.append(1.0 - dp); vol_wts.append(1.0)
    if "flow_duration" in m and "bins_log10" in m["flow_duration"]:
        td = _log10_cdf_on_bins(df_norm["duration"], m["flow_duration"]["bins_log10"])
        dd = ks_distance(m["flow_duration"]["cdf"], td); vol_parts.append(1.0 - dd); vol_wts.append(1.0)
    TR_vol = float(np.average(vol_parts, weights=vol_wts)) if vol_wts else np.nan

    # TR_tmp: service-aware temporal (now with debug + dataset name)
    TR_tmp = score_temporal_service_aware(
        df_norm, m,
        debug=(dataset_name in DEBUG_SERVICES),
        dataset_name=dataset_name
    )

    # ensure canonical services before TR_pro
    if "service" in df_norm.columns:
        df_norm["service"] = canonicalize_service_series(df_norm["service"])

    # TR_pro: protocol/service TV similarity
    TR_pros = []
    if "protocol_mix" in m and df_norm.get("proto") is not None:
        p_ref = m["protocol_mix"]; p_test = mix(df_norm["proto"])
        if p_test: TR_pros.append(1.0 - tv_distance(p_ref, p_test))
    if "service_mix" in m and df_norm.get("service") is not None:
        s_ref = m["service_mix"]; s_test = mix(df_norm["service"])
        if s_test: TR_pros.append(1.0 - tv_distance(s_ref, s_test))
    TR_pro = float(np.mean(TR_pros)) if TR_pros else np.nan

    # Combine with renormalization over available parts
    a,b,g = weights["alpha"], weights["beta"], weights["gamma"]
    parts, wts = [], []
    if not np.isnan(TR_vol): parts.append(TR_vol); wts.append(a)
    if not np.isnan(TR_tmp): parts.append(TR_tmp); wts.append(b)
    if not np.isnan(TR_pro): parts.append(TR_pro); wts.append(g)
    TRI = float(np.dot(parts, np.array(wts)/np.sum(wts))) if wts else np.nan

    return {"TR_vol": TR_vol, "TR_tmp": TR_tmp, "TR_pro": TR_pro, "TRI": TRI}

# -------------- UNR-IDD LOSO (counters) --------------
def unridd_load_normalize_aggregate(path_glob: str) -> pd.DataFrame:
    frames=[]
    for p in _match_paths(path_glob):
        try: frames.append(pd.read_csv(p, low_memory=False))
        except Exception as e: print(f"[WARN] {p}: {e}")
    if not frames: raise FileNotFoundError("No UNR-IDD CSVs matched.")
    raw = pd.concat(frames, ignore_index=True)
    nm = _ci(raw)

    c_sw   = _col(nm,"Switch ID","switch id","switch_id")
    c_port = _col(nm,"Port Number","port number","port_number","port")
    c_winS = _col(nm,"Delta Port alive Duration (S)","Duration (S)","window_s")
    c_rxB  = _col(nm,"Delta Received Bytes")
    c_txB  = _col(nm,"Delta Sent Bytes")
    c_rxP  = _col(nm,"Delta Received Packets")
    c_txP  = _col(nm,"Delta Sent Packets")
    c_bin  = _col(nm,"Binary Label","binary label","binary_label")
    c_lab  = _col(nm,"Label","label")
    if not all([c_sw,c_port,c_winS,c_rxB,c_txB,c_rxP,c_txP]):
        raise ValueError("UNR-IDD: missing counters columns.")

    # benign mask
    if c_bin:
        bl = raw[c_bin].astype(str).str.strip().str.lower()
        benign = bl.isin(["0","normal","benign"])
    elif c_lab:
        lb = raw[c_lab].astype(str).str.strip().str.lower()
        benign = lb.isin(["normal","benign","0","ok","clean","norm"])
    else:
        raise ValueError("UNR-IDD: no labels found.")
    if benign.sum()==0: raise ValueError("UNR-IDD: no benign rows.")

    dfb = pd.DataFrame({
        "switch_id": raw[c_sw],
        "port": raw[c_port],
        "win_s": _to_num(raw[c_winS]),
        "DeltaBytes": _to_num(raw[c_rxB]).fillna(0)+_to_num(raw[c_txB]).fillna(0),
        "DeltaPkts":  _to_num(raw[c_rxP]).fillna(0)+_to_num(raw[c_txP]).fillna(0),
        "benign": benign
    })
    dfb = dfb[(dfb["benign"]) & (dfb["win_s"]>0)].dropna(subset=["switch_id","port","win_s"])

    # aggregate ~5s → ~60s per (switch,port); build pseudo end_ts for burstiness
    rows=[]
    tgt=COUNTERS_TARGET_WINDOW_S; tol=COUNTERS_TOLERANCE_S
    for (sw,pt), g in dfb.groupby(["switch_id","port"], sort=False):
        acc_t=0.0; acc_b=0.0; acc_p=0.0; pseudo_ts=0.0
        for _,r in g.iterrows():
            w=float(r["win_s"])
            if w<=0: continue
            acc_t+=w; acc_b+=float(r["DeltaBytes"]); acc_p+=float(r["DeltaPkts"]); pseudo_ts+=w
            if acc_t>=tgt:
                if (tgt-tol)<=acc_t<=(tgt+tol):
                    rows.append([sw,pt,acc_t,acc_b,acc_p,pseudo_ts])
                acc_t=0.0; acc_b=0.0; acc_p=0.0
    agg = pd.DataFrame(rows, columns=["switch_id","port","win_s","DeltaBytes","DeltaPkts","end_ts"])
    return agg

def _log10_hist_cdf(series: pd.Series, n_bins=N_BINS):
    s = _winsor(series); s = s[s>0]
    if s.empty: return [], []
    logv = np.log10(s.values)
    hist, edges = np.histogram(logv, bins=n_bins, density=True)
    cdf = np.cumsum(hist)/np.sum(hist)
    return edges.tolist(), cdf.tolist()

def _cdf_on_bins_log10(series, bins):
    s = _winsor(series); s = s[s>0]
    if s.empty: return [0.0]*(len(bins)-1)
    logv=np.log10(s.values)
    h,_=np.histogram(logv, bins=np.array(bins), density=True)
    return (np.cumsum(h)/np.sum(h)).tolist() if np.sum(h)>0 else [0.0]*(len(bins)-1)

def _cdf_on_bins_lin(series, bins):
    s = _winsor(series)
    if s.empty: return [0.0]*(len(bins)-1)
    h,_=np.histogram(s.values, bins=np.array(bins), density=True)
    return (np.cumsum(h)/np.sum(h)).tolist() if np.sum(h)>0 else [0.0]*(len(bins)-1)

def ks(c1, c2):
    if (not c1) or (len(c1)!=len(c2)): return 1.0
    return float(np.max(np.abs(np.array(c1)-np.array(c2))))

def build_counters_metrics(train: pd.DataFrame) -> Dict:
    bins_b, cdf_b = _log10_hist_cdf(train["DeltaBytes"], N_BINS)
    bins_p, cdf_p = _log10_hist_cdf(train["DeltaPkts"],  N_BINS)
    metrics = {
        "bytes_per_window": {"bins_log10": bins_b, "cdf": cdf_b},
        "pkts_per_window":  {"bins_log10": bins_p, "cdf": cdf_p},
    }
    # window duration (linear)
    w = _winsor(train["win_s"])
    if not w.empty:
        h,e = np.histogram(w.values, bins=N_BINS, density=True)
        if np.sum(h)>0:
            metrics["window_duration"] = {"bins": e.tolist(), "cdf": (np.cumsum(h)/np.sum(h)).tolist()}
    # burstiness via Fano over 10-min buckets on pseudo-time
    t = pd.to_numeric(train["end_ts"], errors="coerce").dropna()
    if not t.empty:
        t0=t.min(); bucket=np.floor((t-t0)/600.0).astype(int)
        counts=pd.Series(1,index=bucket).groupby(level=0).sum().astype(float)
        lam=counts.mean(); var=counts.var(ddof=0)
        if lam>0: metrics["burstiness"]={"fano_ref": float(var/lam), "bucket_s": 600}
    return metrics

def score_counters_test(test: pd.DataFrame, metrics: Dict, alpha=0.7, beta=0.3) -> Tuple[float,float,float]:
    tb = _cdf_on_bins_log10(test["DeltaBytes"], metrics["bytes_per_window"]["bins_log10"])
    tp = _cdf_on_bins_log10(test["DeltaPkts"],  metrics["pkts_per_window"]["bins_log10"])
    TR_vol = np.mean([1.0-ks(metrics["bytes_per_window"]["cdf"], tb),
                      1.0-ks(metrics["pkts_per_window"]["cdf"],  tp)])
    tmp_parts=[]
    if "window_duration" in metrics:
        tw = _cdf_on_bins_lin(test["win_s"], metrics["window_duration"]["bins"])
        tmp_parts.append(1.0-ks(metrics["window_duration"]["cdf"], tw))
    if "burstiness" in metrics:
        bucket_s = int(metrics["burstiness"].get("bucket_s", 600))
        t = pd.to_numeric(test["end_ts"], errors="coerce").dropna()
        if not t.empty:
            t0=t.min(); bucket=np.floor((t-t0)/bucket_s).astype(int)
            counts=pd.Series(1,index=bucket).groupby(level=0).sum().astype(float)
            lam=counts.mean(); var=counts.var(ddof=0)
            if lam>0:
                f_test=float(var/lam); f_ref=float(metrics["burstiness"]["fano_ref"])
                sim=float(np.exp(-abs(np.log((f_test+1e-9)/(f_ref+1e-9)))))
                tmp_parts.append(sim)
    TR_tmp = float(np.mean(tmp_parts)) if tmp_parts else np.nan
    TRI = TR_vol if np.isnan(TR_tmp) else (alpha*TR_vol + beta*TR_tmp)
    return TR_vol, TR_tmp, TRI

def unridd_loso_tri(path_glob: str, weights=WEIGHTS["counters"]) -> Dict[str,float]:
    agg = unridd_load_normalize_aggregate(path_glob)
    if agg.empty:
        raise SystemExit("UNR-IDD: no ~60s windows produced.")
    switches = sorted(agg["switch_id"].unique().tolist())
    rows=[]
    for sw in switches:
        test  = agg[agg["switch_id"]==sw]
        train = agg[agg["switch_id"]!=sw]
        if train.empty or test.empty: continue
        metrics = build_counters_metrics(train)
        TR_vol, TR_tmp, TRI = score_counters_test(test, metrics, alpha=weights["alpha"], beta=weights["beta"])
        rows.append({"switch_id": sw, "rows_test": len(test), "TR_vol": TR_vol, "TR_tmp": TR_tmp, "TRI": TRI})
    if not rows:
        return {"TR_vol": np.nan, "TR_tmp": np.nan, "TRI": np.nan, "rows_used": 0}
    res = pd.DataFrame(rows)
    w = res["rows_test"].values
    return {
        "TR_vol": float(np.average(res["TR_vol"], weights=w)),
        "TR_tmp": float(np.average(res["TR_tmp"], weights=w)) if res["TR_tmp"].notna().any() else np.nan,
        "TRI":    float(np.average(res["TRI"],    weights=w)),
        "rows_used": int(res["rows_test"].sum())
    }

# -------------- run all --------------
NORMALIZERS = {
    "NSL-KDD": norm_nsl_kdd,
    "UNSW-NB15": norm_unsw,
    "CIC-IDS2017": norm_cic2017,
    "CSE-CIC-IDS2018": norm_cse_cic_2018,
    "ToN-IoT": norm_ton_iot,
    "BoT-IoT": norm_bot_iot,
    "HIKARI-2021": norm_hikari,
}

rows = []

# First 7 datasets via baselines
for name in ["NSL-KDD","UNSW-NB15","CIC-IDS2017","CSE-CIC-IDS2018","ToN-IoT","BoT-IoT","HIKARI-2021"]:
    profile_key = PROFILE_MAP[name]
    bl_path = BASELINES.get(profile_key)
    if not bl_path or not os.path.exists(bl_path):
        print(f"[ERROR] Baseline not found for {name} (profile={profile_key}): {bl_path}")
        continue
    baseline = load_baseline(bl_path)
    weights  = WEIGHTS[profile_key]

    print(f"\n[INFO] {name}: loading…")
    df_raw = load_csv(DATASETS[name])
    df_norm = NORMALIZERS[name](df_raw)
    # normalize service labels for overlap with baseline
    if "service" in df_norm.columns:
        df_norm["service"] = canonicalize_service_series(df_norm["service"])
    sc = score_flow_like(df_norm, baseline, weights, dataset_name=name)
    rows.append({"dataset": name, "TR_vol": sc["TR_vol"], "TR_tmp": sc["TR_tmp"], "TR_pro": sc["TR_pro"], "TRI": sc["TRI"], "rows_used": len(df_norm)})

# UNR-IDD via LOSO (no external baseline)
print(f"\n[INFO] UNR-IDD: LOSO scoring…")
loso = unridd_loso_tri(DATASETS["UNR-IDD"], weights=WEIGHTS["counters"])
rows.append({"dataset": "UNR-IDD", "TR_vol": loso["TR_vol"], "TR_tmp": loso["TR_tmp"], "TR_pro": np.nan, "TRI": loso["TRI"], "rows_used": loso["rows_used"]})


# ---------- TRI output directory ----------
TRI_OUTDIR = "tri_outputs"
os.makedirs(TRI_OUTDIR, exist_ok=True)


# Results table
res = pd.DataFrame(rows)

def f(x):
    try: return f"{x:.6f}"
    except: return x

if not res.empty:
    print("\nScores (context-aware, UNR-IDD via LOSO):")
    cols = ["dataset","TR_vol","TR_tmp","TR_pro","TRI","rows_used"]
    print(res[cols].map(lambda x: f(x) if isinstance(x,(float,np.floating)) else x)
              .to_string(index=False))

    # NEW: save dataset-level TRI summary
    summary_path = os.path.join(TRI_OUTDIR, "tri_dataset_summary.csv")
    res[cols].to_csv(summary_path, index=False)
    print(f"[EXPORT] TRI dataset summary written: {summary_path}")
else:
    print("[WARN] No scores computed. Check paths/baselines].")


# --- Export service-aware diagnostics as CSVs ---
try:
    ts_tag = datetime.utcnow().strftime("%Y%m%d_%H%M%S")

    if OVERLAP_ROWS:
        df_overlap = pd.DataFrame(OVERLAP_ROWS)
        overlap_path = os.path.join(
            TRI_OUTDIR, f"tri_tmp_service_overlap_{ts_tag}.csv"
        )
        df_overlap.to_csv(overlap_path, index=False)
        print(f"\n[EXPORT] Service overlap written: {overlap_path}  (rows={len(df_overlap)})")
    else:
        print("\n[EXPORT] No service overlap rows collected.")

    if CONTRIB_ROWS:
        df_contrib = pd.DataFrame(CONTRIB_ROWS)
        df_contrib = df_contrib.sort_values(by=["dataset","weight"], ascending=[True, False])
        contrib_path = os.path.join(
            TRI_OUTDIR, f"tri_tmp_service_contrib_{ts_tag}.csv"
        )
        df_contrib.to_csv(contrib_path, index=False)
        print(f"[EXPORT] Service contributions written: {contrib_path}  (rows={len(df_contrib)})")
    else:
        print("[EXPORT] No service contribution rows collected.")
except Exception as e:
    print(f"[WARN] Could not export service diagnostics: {e}")



[INFO] NSL-KDD: loading…

[INFO] UNSW-NB15: loading…

[INFO] CIC-IDS2017: loading…

[INFO] CSE-CIC-IDS2018: loading…

[INFO] ToN-IoT: loading…

[INFO] BoT-IoT: loading…

[INFO] HIKARI-2021: loading…

[INFO] UNR-IDD: LOSO scoring…

Scores (context-aware, UNR-IDD via LOSO):
        dataset   TR_vol   TR_tmp   TR_pro      TRI  rows_used
        NSL-KDD 0.592192      nan 0.366684 0.517023     148515
      UNSW-NB15 0.563278 0.443839 0.371476 0.477142     257673
    CIC-IDS2017 0.548672 0.574692 0.408353 0.531016    2830743
CSE-CIC-IDS2018 0.545935 0.572357 0.405161 0.528349   16233002
        ToN-IoT 0.530953 0.347372 0.531465 0.457623     211043
        BoT-IoT 0.239529 0.659136 0.357336 0.430933    3668522
    HIKARI-2021 0.498296 0.415157 0.569839 0.479349     555278
        UNR-IDD 0.846549 0.558494      nan 0.760132        282
[EXPORT] TRI dataset summary written: tri_outputs\tri_dataset_summary.csv

[EXPORT] Service overlap written: tri_outputs\tri_tmp_service_overlap_20260109_16412

In [8]:
#Some Diagnostics

## Compare reference and test log10-CDF for bytes for BoT-IoT
bl_bot = load_baseline(BASELINES[PROFILE_MAP["BoT-IoT"]])
m_bot = bl_bot["metrics"]
df_bot = load_csv(DATASETS["BoT-IoT"])
dn = NORMALIZERS["BoT-IoT"](df_bot)
if "bytes_per_flow" in m_bot:
    test_cdf = _log10_cdf_on_bins(dn["bytes"], m_bot["bytes_per_flow"]["bins_log10"])
    print("BoT-IoT: ref bytes bins_len=", len(m_bot["bytes_per_flow"]["bins_log10"]), 
          " test cdf_len=", len(test_cdf))
    # optionally print a few cdf points
    print("ref cdf (first 6):", m_bot["bytes_per_flow"]["cdf"][:6])
    print("test cdf (first 6):", test_cdf[:6])


for name in ["NSL-KDD","UNSW-NB15","CIC-IDS2017","CSE-CIC-IDS2018","ToN-IoT","BoT-IoT","HIKARI-2021"]:
    df_raw = load_csv(DATASETS[name])
    df_norm = NORMALIZERS[name](df_raw)
    print(f"{name}: rows={len(df_norm)}, start_ts_nonnull={df_norm['start_ts'].notna().sum() if 'start_ts' in df_norm.columns else 'n/a'}, "
          f"flow_iat_mean_nonnull={df_norm.get('flow_iat_mean_vals').notna().sum() if 'flow_iat_mean_vals' in df_norm.columns else 'n/a'}")

# load baseline for HIKARI profile
bl = load_baseline(BASELINES[PROFILE_MAP["HIKARI-2021"]])
m = bl["metrics"]
print("HIKARI baseline services (top):", sorted(m.get("service_mix", {}).items(), key=lambda x: x[1], reverse=True)[:12])

# inspect HIKARI dataset services
df_h = load_csv(DATASETS["HIKARI-2021"])
dhn = NORMALIZERS["HIKARI-2021"](df_h)
if "service" in dhn.columns:
    s = canonicalize_service_series(dhn["service"])
    vc = s.dropna().astype(str).value_counts(normalize=True)
    print("HIKARI top services (test):", vc.head(12).to_dict())
else:
    print("HIKARI: no service column present after normalization.")

# Compare reference and test log10-CDF for bytes for BoT-IoT
bl_bot = load_baseline(BASELINES[PROFILE_MAP["BoT-IoT"]])
m_bot = bl_bot["metrics"]
df_bot = load_csv(DATASETS["BoT-IoT"])
dn = NORMALIZERS["BoT-IoT"](df_bot)
if "bytes_per_flow" in m_bot:
    test_cdf = _log10_cdf_on_bins(dn["bytes"], m_bot["bytes_per_flow"]["bins_log10"])
    print("BoT-IoT: ref bytes bins_len=", len(m_bot["bytes_per_flow"]["bins_log10"]), 
          " test cdf_len=", len(test_cdf))
    # optionally print a few cdf points
    print("ref cdf (first 6):", m_bot["bytes_per_flow"]["cdf"][:6])
    print("test cdf (first 6):", test_cdf[:6])



BoT-IoT: ref bytes bins_len= 51  test cdf_len= 50
ref cdf (first 6): [0.00012556701354554146, 0.00012556701354554146, 0.00012556701354554146, 0.00012556701354554146, 0.00012556701354554146, 0.00012556701354554146]
test cdf (first 6): [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
NSL-KDD: rows=148515, start_ts_nonnull=0, flow_iat_mean_nonnull=n/a
UNSW-NB15: rows=257673, start_ts_nonnull=0, flow_iat_mean_nonnull=257673
CIC-IDS2017: rows=2830743, start_ts_nonnull=0, flow_iat_mean_nonnull=2830743
CSE-CIC-IDS2018: rows=16233002, start_ts_nonnull=0, flow_iat_mean_nonnull=16232943
ToN-IoT: rows=211043, start_ts_nonnull=0, flow_iat_mean_nonnull=211043
BoT-IoT: rows=3668522, start_ts_nonnull=3668522, flow_iat_mean_nonnull=n/a
HIKARI-2021: rows=555278, start_ts_nonnull=0, flow_iat_mean_nonnull=555278
HIKARI baseline services (top): [('dns', 0.6807592082798958), ('https', 0.1698661133666076), ('http', 0.07231838715211122), ('ntp', 0.07118700834034489), ('dhcp', 0.005342395255560911), ('quic,ssl', 0.000416608804

In [22]:
# TRI_visualization_helpers
# Requires: numpy, pandas, matplotlib, yaml, os

import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# ---------- Config ----------
OUTDIR = "tri_visuals"
os.makedirs(OUTDIR, exist_ok=True)

# If True, recompute TRI scores from scratch using your scoring functions.
RECOMPUTE = False

# Filenames for baseline YAMLs (should match variables in your session)
BASELINE_FILES = {
    "enterprise": r"baselines/general_enterprise_nids_baseline_20251016.yaml",
    "iot":        r"baselines/iot_gateway_nids_baseline_20251017.yaml",
    "encrypted_flow_only_ids": r"baselines/encrypted_flow_only_ids_baseline_20251129.yaml",
}

# Dataset ordering for plots (will use keys in your DATASETS dict)
DATASET_ORDER = list(DATASETS.keys())  # uses existing DATASETS mapping


# ---------- helper plotting routines ----------
def _safe_mkdir(path):
    if not os.path.exists(path):
        os.makedirs(path, exist_ok=True)

def plot_tri_summary(res_df: pd.DataFrame, outdir: str = OUTDIR, fname="tri_summary_bar.png"):
    """
    Grouped bar plot: TR_vol, TR_tmp, TR_pro (where available) + TRI as line overlay.
    Expects columns: dataset, TR_vol, TR_tmp, TR_pro, TRI
    """
    _safe_mkdir(outdir)
    df = res_df.copy()
    df = df.set_index("dataset").reindex(DATASET_ORDER).reset_index()
    n = len(df)
    idx = np.arange(n)

    # Build arrays (use nan->0 for bar heights but mask)
    vol = df["TR_vol"].fillna(np.nan).to_numpy(dtype=float)
    tmp = df["TR_tmp"].fillna(np.nan).to_numpy(dtype=float)
    pro = df["TR_pro"].fillna(np.nan).to_numpy(dtype=float)
    tri = df["TRI"].fillna(np.nan).to_numpy(dtype=float)

    width = 0.20
    fig, ax = plt.subplots(figsize=(max(8, n*0.8), 6))

    x = idx - width
    any_bars = False
    if not np.all(np.isnan(vol)):
        ax.bar(x, np.nan_to_num(vol, 0.0), width=width, label="TR_vol")
        any_bars = True
    if not np.all(np.isnan(tmp)):
        ax.bar(x + width, np.nan_to_num(tmp, 0.0), width=width, label="TR_tmp")
        any_bars = True
    if not np.all(np.isnan(pro)):
        ax.bar(x + 2*width, np.nan_to_num(pro, 0.0), width=width, label="TR_pro")
        any_bars = True

    # TRI line (plotted on same axis)
    ax.plot(idx, np.nan_to_num(tri, np.nan), marker="o", linestyle="-", linewidth=2, label="TRI")
    ax.set_xticks(idx)
    ax.set_xticklabels(df["dataset"], rotation=45, ha="right")
    ax.set_ylabel("Score (0..1)")
    ax.set_title("TRI summary: components + TRI")
    ax.grid(axis="y", alpha=0.2)
    ax.legend()
    fig.tight_layout()
    out_path = os.path.join(outdir, fname)
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    print("[VIS] Saved TRI summary:", out_path)


def _plot_log10_cdf_overlay(bins_ref, cdf_ref, series_test, outpath, title="bytes per flow", xlabel="log10(value)"):
    """
    Plot baseline CDF (ref) and test CDF computed on ref bins.
    bins_ref: edges array of length B (returned by baseline)
    cdf_ref: length B-1
    series_test: pandas Series (raw values)
    """
    # compute test cdf on same bins
    tb = _log10_cdf_on_bins(series_test, bins_ref)
    # x positions (center of bins)
    edges = np.array(bins_ref)
    centers = 0.5 * (edges[:-1] + edges[1:])
    fig, ax = plt.subplots(figsize=(7,4))
    ax.step(centers, cdf_ref, where="mid", label="baseline CDF")
    ax.step(centers, tb, where="mid", label="test CDF")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("CDF")
    ax.set_title(title)
    ax.set_ylim(-0.02,1.02)
    ax.grid(alpha=0.2)
    ax.legend()
    fig.tight_layout()
    fig.savefig(outpath, dpi=150)
    plt.close(fig)


def _plot_linear_cdf_overlay(bins_ref, cdf_ref, series_test, outpath, title="window duration", xlabel="value"):
    # compute test cdf via lin bins
    tw = _lin_cdf_on_bins(series_test, bins_ref)
    edges = np.array(bins_ref)
    centers = 0.5 * (edges[:-1] + edges[1:])
    fig, ax = plt.subplots(figsize=(7,4))
    ax.step(centers, cdf_ref, where="mid", label="baseline CDF")
    ax.step(centers, tw, where="mid", label="test CDF")
    ax.set_xlabel(xlabel); ax.set_ylabel("CDF"); ax.set_title(title)
    ax.set_ylim(-0.02,1.02); ax.grid(alpha=0.2); ax.legend()
    fig.tight_layout(); fig.savefig(outpath, dpi=150); plt.close(fig)


def _bar_compare_dicts(d_ref: dict, d_test: dict, outpath, title="protocol mix", top_k=10):
    # take union of keys, but plot top_k of combined
    keys = set(d_ref.keys()) | set(d_test.keys())
    # order by max share
    order = sorted(list(keys), key=lambda k: max(d_ref.get(k,0.0), d_test.get(k,0.0)), reverse=True)[:top_k]
    vals_ref = [d_ref.get(k,0.0) for k in order]
    vals_test = [d_test.get(k,0.0) for k in order]
    idx = np.arange(len(order))
    width = 0.35
    fig, ax = plt.subplots(figsize=(max(6,len(order)*0.6),4))
    ax.bar(idx - width/2, vals_ref, width=width, label="baseline")
    ax.bar(idx + width/2, vals_test, width=width, label="test")
    ax.set_xticks(idx); ax.set_xticklabels(order, rotation=45, ha="right")
    ax.set_ylabel("share"); ax.set_title(title); ax.legend(); ax.grid(axis="y", alpha=0.2)
    fig.tight_layout(); fig.savefig(outpath, dpi=150); plt.close(fig)


# ---------- baseline vs dataset comparison ----------
def compare_baseline_to_dataset(baseline: dict, baseline_key: str,
                                dataset_name: str, dataset_path: str,
                                normalizer_fn,
                                outdir_base: str = OUTDIR,
                                save_temporal_csv: bool = True):
    """
    Generate per-metric comparison plots for one baseline vs one dataset.
    baseline: loaded YAML dict
    normalizer_fn: e.g., NORMALIZERS["HIKARI-2021"]
    """
    ds_outdir = os.path.join(outdir_base, f"baseline_{baseline_key}", dataset_name)
    _safe_mkdir(ds_outdir)

    # Load dataset and normalize (if normalizer provided)
    raw = load_csv(dataset_path)
    df_norm = normalizer_fn(raw) if callable(normalizer_fn) else raw.copy()

    metrics = baseline.get("metrics", {})

    # Bytes, pkts, duration (log10 bins)
    if "bytes_per_flow" in metrics and "bytes" in df_norm.columns:
        outp = os.path.join(ds_outdir, f"bytes_cdf_{dataset_name}.png")
        _plot_log10_cdf_overlay(metrics["bytes_per_flow"]["bins_log10"],
                                metrics["bytes_per_flow"]["cdf"],
                                df_norm["bytes"], outp,
                                title=f"bytes per flow: {baseline_key} vs {dataset_name}", xlabel="log10(bytes)")
    if "pkts_per_flow" in metrics and "pkts" in df_norm.columns:
        outp = os.path.join(ds_outdir, f"pkts_cdf_{dataset_name}.png")
        _plot_log10_cdf_overlay(metrics["pkts_per_flow"]["bins_log10"],
                                metrics["pkts_per_flow"]["cdf"],
                                df_norm["pkts"], outp,
                                title=f"pkts per flow: {baseline_key} vs {dataset_name}", xlabel="log10(pkts)")
    if "flow_duration" in metrics and "duration" in df_norm.columns:
        outp = os.path.join(ds_outdir, f"duration_cdf_{dataset_name}.png")
        _plot_log10_cdf_overlay(metrics["flow_duration"]["bins_log10"],
                                metrics["flow_duration"]["cdf"],
                                df_norm["duration"], outp,
                                title=f"flow duration: {baseline_key} vs {dataset_name}", xlabel="log10(seconds)")

    # iat (if baseline has iat info)
    if "iat_flows" in metrics:
        if "start_ts" in df_norm.columns:
            outp = os.path.join(ds_outdir, f"iat_cdf_{dataset_name}.png")
            _plot_log10_cdf_overlay(metrics["iat_flows"]["bins_log10"],
                                    metrics["iat_flows"]["cdf"],
                                    # compute inter-flow iat global for test
                                    pd.to_numeric(df_norm["start_ts"], errors="coerce").dropna().sort_values().diff().dropna(),
                                    outp,
                                    title=f"inter-flow IAT (global): {baseline_key} vs {dataset_name}", xlabel="log10(seconds)")
        elif "flow_iat_mean_vals" in df_norm.columns and "flow_iat_mean" in metrics:
            outp = os.path.join(ds_outdir, f"flow_iat_mean_cdf_{dataset_name}.png")
            _plot_log10_cdf_overlay(metrics["flow_iat_mean"]["bins_log10"],
                                    metrics["flow_iat_mean"]["cdf"],
                                    pd.to_numeric(df_norm["flow_iat_mean_vals"], errors="coerce").dropna(),
                                    outp,
                                    title=f"within-flow IAT mean: {baseline_key} vs {dataset_name}", xlabel="log10(seconds)")

    # Protocol mix
    if "protocol_mix" in metrics and "proto" in df_norm.columns:
        p_ref = metrics["protocol_mix"]
        p_test = mix(df_norm["proto"]) or {}
        outp = os.path.join(ds_outdir, f"protocol_mix_{dataset_name}.png")
        _bar_compare_dicts(p_ref, p_test, outp, title=f"Protocol mix: {baseline_key} vs {dataset_name}")

    # Service mix
    if "service_mix" in metrics:
        s_ref = metrics["service_mix"]
        s_test = mix(df_norm.get("service")) or {}
        outp = os.path.join(ds_outdir, f"service_mix_{dataset_name}.png")
        _bar_compare_dicts(s_ref, s_test, outp, title=f"Service mix: {baseline_key} vs {dataset_name}")

    # temporal_by_service: compute overlap table and save csv
    if "temporal_by_service" in metrics and "service" in df_norm.columns:
        svc_all = canonicalize_service_series(df_norm["service"])
        s_test = svc_all.dropna().astype(str).value_counts(normalize=True).to_dict()
        s_ref = {canonicalize_service_name(k): v.get("share", 0.0) for k,v in metrics["temporal_by_service"].items()}
        # create overlap dataframe
        keys = sorted(set(s_ref.keys()) | set(s_test.keys()), key=lambda k: s_test.get(k,0.0), reverse=True)
        rows = []
        for k in keys:
            rows.append({"service": k, "ref_share": float(s_ref.get(k,0.0)), "test_share": float(s_test.get(k,0.0))})
        df_ov = pd.DataFrame(rows)
        csvp = os.path.join(ds_outdir, f"temporal_by_service_{dataset_name}.csv")
        df_ov.to_csv(csvp, index=False)
        print("[VIS] Saved temporal_by_service overlap ->", csvp)

    print(f"[VIS] Completed baseline '{baseline_key}' vs dataset '{dataset_name}' -> outputs in {ds_outdir}")


# ---------- master routine ----------
def visualize_all_tri(outdir: str = OUTDIR, recompute: bool = RECOMPUTE):
    _safe_mkdir(outdir)
    # 1) compute or load the res DataFrame of TRI scores
    if recompute:
        print("[VIS] Recomputing TRI scores for datasets...")
        rows_local = []
        # first 7 datasets as in your pipeline
        for name in DATASET_ORDER:
            # skip UNR-IDD for baseline-driven scoring (it uses LOSO)
            if name == "UNR-IDD":
                continue
            profile = PROFILE_MAP.get(name, None)
            if not profile:
                print("[WARN] No profile for", name); continue
            bl_path = BASELINE_FILES.get(profile)
            if not bl_path or not os.path.exists(bl_path):
                print(f"[ERROR] Baseline not found for {name} (profile={profile}): {bl_path}")
                rows_local.append({"dataset": name, "TR_vol": np.nan, "TR_tmp": np.nan, "TR_pro": np.nan, "TRI": np.nan, "rows_used": 0})
                continue
            baseline = load_baseline(bl_path)
            weights = WEIGHTS[profile]
            df_raw = load_csv(DATASETS[name])
            df_norm = NORMALIZERS[name](df_raw)
            if "service" in df_norm.columns:
                df_norm["service"] = canonicalize_service_series(df_norm["service"])
            sc = score_flow_like(df_norm, baseline, weights, dataset_name=name)
            rows_local.append({"dataset": name,
                               "TR_vol": sc["TR_vol"], "TR_tmp": sc["TR_tmp"], "TR_pro": sc["TR_pro"],
                               "TRI": sc["TRI"], "rows_used": len(df_norm)})
        # UNR-IDD via LOSO
        loso_res = unridd_loso_tri(DATASETS["UNR-IDD"], weights=WEIGHTS["counters"])
        rows_local.append({"dataset": "UNR-IDD", "TR_vol": loso_res["TR_vol"], "TR_tmp": loso_res["TR_tmp"],
                           "TR_pro": np.nan, "TRI": loso_res["TRI"], "rows_used": loso_res["rows_used"]})
        res_df = pd.DataFrame(rows_local)
    else:
        # try to pick up 'res' from global namespace
        try:
            res_df = globals().get("res")
            if res_df is None:
                raise NameError("res not found")
            # ensure dataset column exists
            if "dataset" not in res_df.columns:
                res_df = res_df.reset_index().rename(columns={"index":"dataset"})
        except Exception as e:
            raise RuntimeError("No existing TRI results found in session; set recompute=True to recompute.") from e

    # 2) Plot TRI summary
    plot_tri_summary(res_df, outdir=outdir)

    # 3) For each baseline, list datasets that used it (PROFILE_MAP) and run comparisons
    for baseline_key, baseline_path in BASELINE_FILES.items():
        if not baseline_path or not os.path.exists(baseline_path):
            print(f"[VIS] skipping baseline {baseline_key}: file not found -> {baseline_path}")
            continue
        baseline = load_baseline(baseline_path)
        # identify datasets that map to this profile_key
        ds_for_baseline = [d for d, p in PROFILE_MAP.items() if p == baseline_key]
        if not ds_for_baseline:
            print(f"[VIS] no datasets assigned to baseline '{baseline_key}' (profile map).")
            continue
        for ds in ds_for_baseline:
            # get data path and normalizer
            dp = DATASETS.get(ds)
            norm_fn = NORMALIZERS.get(ds)
            if dp is None or norm_fn is None:
                print(f"[VIS] skip {ds}: missing DATASETS/NORMALIZERS entry.")
                continue
            try:
                compare_baseline_to_dataset(baseline=baseline, baseline_key=baseline_key,
                                            dataset_name=ds, dataset_path=dp,
                                            normalizer_fn=norm_fn, outdir_base=outdir)
            except Exception as e:
                print(f"[WARN] compare failed for baseline {baseline_key} vs {ds}: {e}")

    print("[VIS] All visualizations done. Output directory:", outdir)


# ---------- usage ----------
# 1) If you already have 'res' DataFrame in memory (from previous scoring), call:
#    visualize_all_tri(outdir="tri_visuals", recompute=False)
# 2) If you want to recompute TRI scores first, call:
#    visualize_all_tri(outdir="tri_visuals", recompute=True)
#
# Run one of the two below:
visualize_all_tri(outdir=OUTDIR, recompute=False)
# OR
# visualize_all_tri(outdir=OUTDIR, recompute=True)


[VIS] Saved TRI summary: tri_visuals\tri_summary_bar.png
[VIS] Saved temporal_by_service overlap -> tri_visuals\baseline_enterprise\NSL-KDD\temporal_by_service_NSL-KDD.csv
[VIS] Completed baseline 'enterprise' vs dataset 'NSL-KDD' -> outputs in tri_visuals\baseline_enterprise\NSL-KDD
[VIS] Saved temporal_by_service overlap -> tri_visuals\baseline_enterprise\UNSW-NB15\temporal_by_service_UNSW-NB15.csv
[VIS] Completed baseline 'enterprise' vs dataset 'UNSW-NB15' -> outputs in tri_visuals\baseline_enterprise\UNSW-NB15
[VIS] Saved temporal_by_service overlap -> tri_visuals\baseline_enterprise\CIC-IDS2017\temporal_by_service_CIC-IDS2017.csv
[VIS] Completed baseline 'enterprise' vs dataset 'CIC-IDS2017' -> outputs in tri_visuals\baseline_enterprise\CIC-IDS2017
[VIS] Saved temporal_by_service overlap -> tri_visuals\baseline_enterprise\CSE-CIC-IDS2018\temporal_by_service_CSE-CIC-IDS2018.csv
[VIS] Completed baseline 'enterprise' vs dataset 'CSE-CIC-IDS2018' -> outputs in tri_visuals\baseline_en